# **SpaceX Falcon 9 first stage Landing Prediction**
# Lab 1: Collecting the data (API SpaceX)

Dans ce capstone, on prédit si le premier étage du Falcon 9 atterrit avec succès. SpaceX annonce un coût de 62 M$ par lancement (contre 165 M$ et plus chez les concurrents), l'économie venant de la réutilisation du premier étage. Prédire l'atterrissage permet donc d'estimer le coût d'un lancement.

**Objectifs de ce notebook**
- Faire des requêtes GET vers l'API SpaceX
- Nettoyer et mettre en forme les données collectées

**Tasks** : (1) requête + parsing JSON, (2) filtrer Falcon 9, (3) traiter les valeurs manquantes.

## Import Libraries and Define Auxiliary Functions

In [1]:
import time
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

### Fonction de requête robuste
L'API publique de SpaceX est parfois instable (erreurs 5xx, pages HTML au lieu de JSON). `safe_get` réessaie plusieurs fois et **met en cache** chaque réponse : une même fusée ou un même pas de tir n'est interrogé qu'une seule fois (moins d'appels, donc moins de risques d'échec).

In [2]:
_cache = {}

def safe_get(url, retries=5, delay=2):
    """GET + JSON avec nouvelles tentatives et cache. Lève RuntimeError si tout échoue."""
    if url in _cache:
        return _cache[url]
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            last = r.status_code
            if r.status_code == 200:
                _cache[url] = r.json()
                return _cache[url]
        except (requests.RequestException, ValueError) as e:
            last = repr(e)
        time.sleep(delay * (attempt + 1))
    raise RuntimeError(f"Echec apres {retries} essais : {url} (dernier statut : {last})")

Depuis la colonne `rocket`, on récupère le nom du booster.

In [3]:
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = safe_get("https://api.spacexdata.com/v4/rockets/" + str(x))
            BoosterVersion.append(response['name'])

Depuis `launchpad`, on récupère le nom du site, la longitude et la latitude.

In [4]:
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = safe_get("https://api.spacexdata.com/v4/launchpads/" + str(x))
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

Depuis `payloads`, on récupère la masse de la charge utile et l'orbite.

In [5]:
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = safe_get("https://api.spacexdata.com/v4/payloads/" + str(load))
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

Depuis `cores`, on récupère le résultat et le type d'atterrissage, le nombre de vols, les ailerons (gridfins), la réutilisation, les jambes (legs), la plateforme, le bloc, le nombre de réutilisations et le numéro de série.

In [6]:
def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = safe_get("https://api.spacexdata.com/v4/cores/" + str(core['core']))
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

Test de l'API en direct (la réponse peut être volumineuse ; on affiche seulement le statut et un extrait).

In [7]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url, timeout=30)
print("Statut :", response.status_code)
print(response.content[:300])

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None))

### Task 1: Request and parse the SpaceX launch data using the GET request
Pour des résultats reproductibles, on utilise la réponse statique fournie par IBM.

In [ ]:
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url)
response.status_code

In [ ]:
# Decoder le JSON et le transformer en DataFrame
data = pd.json_normalize(response.json())

In [ ]:
# Les 5 premieres lignes
data.head()

La plupart des colonnes sont des identifiants. On garde `rocket`, `payloads`, `launchpad`, `cores`, `flight_number` et `date_utc`, puis on filtre.

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']].copy()

# On retire les lancements a plusieurs cores ou plusieurs charges utiles
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# cores et payloads sont des listes de taille 1 : on extrait l'element
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Conversion de date_utc en date
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Restriction des dates
data = data[data['date'] <= datetime.date(2020, 11, 13)].copy()
print(data.shape)

Les résultats des requêtes sont stockés dans des listes (variables globales), puis combinés dans un DataFrame.

In [ ]:
# Variables globales : on (re)initialise TOUJOURS les listes avant de lancer les appels,
# sinon relancer la cellule d'appels duplique les donnees.
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

In [ ]:
BoosterVersion  # vide avant l'appel

Appel des 4 fonctions sur le **même** DataFrame `data`, dans une seule cellule (peut prendre quelques minutes).

In [ ]:
API_OK = True
try:
    getBoosterVersion(data)
    getPayloadData(data)
    getLaunchSite(data)
    getCoreData(data)
    print("Lignes dans data :", len(data))
    print("Longueurs des listes :", len(BoosterVersion), len(PayloadMass), len(Orbit),
          len(LaunchSite), len(Outcome), len(Flights), len(Block), len(Serial))
except RuntimeError as e:
    API_OK = False
    print("API SpaceX indisponible (erreur serveur, pas une erreur de code) :")
    print(e)
    print("-> Reessaie plus tard (relance la cellule des listes vides puis celle-ci),")
    print("   ou continue en MODE SECOURS avec le CSV fourni par IBM (cellules suivantes).")

In [ ]:
BoosterVersion[0:5]  # la liste est maintenant remplie

Construction du jeu de données final à partir des listes.

In [ ]:
if API_OK:
    launch_dict = {'FlightNumber': list(data['flight_number']),
     'Date': list(data['date']),
     'BoosterVersion': BoosterVersion,
     'PayloadMass': PayloadMass,
     'Orbit': Orbit,
     'LaunchSite': LaunchSite,
     'Outcome': Outcome,
     'Flights': Flights,
     'GridFins': GridFins,
     'Reused': Reused,
     'Legs': Legs,
     'LandingPad': LandingPad,
     'Block': Block,
     'ReusedCount': ReusedCount,
     'Serial': Serial,
     'Longitude': Longitude,
     'Latitude': Latitude}

In [ ]:
if API_OK:
    # Creer un DataFrame a partir de launch_dict
    data = pd.DataFrame.from_dict(launch_dict)

In [ ]:
data.head()

### Task 2: Filter the dataframe to only include `Falcon 9` launches

In [ ]:
FALLBACK_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
if API_OK:
    # Garder uniquement les lancements Falcon 9
    data_falcon9 = data[data['BoosterVersion'] != 'Falcon 1'].copy()
else:
    # MODE SECOURS : jeu de donnees IBM (resultat deja nettoye de ce laboratoire)
    data_falcon9 = pd.read_csv(FALLBACK_URL)
    print("MODE SECOURS : API indisponible, utilisation de dataset_part_1.csv (IBM).")
data_falcon9.shape

**Résultat attendu (Task 2)** : `(90, 17)` — 90 lancements Falcon 9, 17 colonnes.

In [ ]:
# Remise a zero de FlightNumber
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9.head()

## Data Wrangling
Certaines lignes ont des valeurs manquantes.

In [ ]:
data_falcon9.isnull().sum()

`LandingPad` garde ses valeurs `None` (aucune plateforme utilisée). Il faut traiter `PayloadMass`.

### Task 3: Dealing with Missing Values

In [ ]:
if API_OK:
    print("NaN avant :", data_falcon9['PayloadMass'].isnull().sum())   # attendu : 5
    payload_mean = data_falcon9['PayloadMass'].mean()
    print("Moyenne PayloadMass :", payload_mean)                        # attendu : ~6123.55
    data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mean)
    print("NaN apres :", data_falcon9['PayloadMass'].isnull().sum())    # attendu : 0
else:
    print("MODE SECOURS : le CSV IBM contient deja la moyenne imputee dans PayloadMass.")
    print("NaN dans PayloadMass :", data_falcon9['PayloadMass'].isnull().sum())
    print("Task 3 (avant/apres) non reproductible sans l'API ; a relancer quand elle est disponible.")

In [ ]:
data_falcon9.isnull().sum()

**Résultat attendu (Task 3)** : moyenne ≈ **6123,55 kg** ; `PayloadMass` passe de **5 à 0** valeur manquante ; `LandingPad` garde **26** valeurs `None` (normal).

## Export
Le CSV servira au notebook suivant (nettoyage / EDA).

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Fichier enregistre : dataset_part_1.csv", data_falcon9.shape)